# Datamine SURLOG Process Example

This notebook demonstrates how to configure and run the **`surlog`** process wrapper in `dmstudio`.

## Process Description

## SURLOG

**SURLOG** is a data input process which will read a character format system file containing records of survey information and measurements as recorded on a digital data recorder by electronic survey equipment.

This process constitutes the first step in survey data processing, creating a Datamine file of the survey measurements. The file stored on the data logger containing the measurements will have to be transferred to the computer, as an ASCII format file, prior to running this process.

The process will create up to three output files of data containing general survey job information, coordinates of the survey stations occupied and referenced during the survey and angle and/or distance measurements taken to fix the position of survey stations or mining excavations. The output files can then be input to suitable survey measurement reduction processes:

* [SURTAC](<surtac.md>)

Reduction of survey tacheometry measurements.

* [SUROBS](<surobs.md>)

Reduction of measurements to survey stations.

It is suitable for the input of survey data, where job header information records are followed by measured angles and distances to surface or underground features. It can also be used for the input of borehole logs where header records containing borehole identifier and collar coordinates are followed by down-the-hole measurements to lithological contacts and mineral assay samples. The data needs to be processed further to compute sample positions in space for geological interpretation work:

* [DESURV](<desurv.md>)

Drillhole survey data is used to produce a file of samples independently located in space.

## * SURLOG

Combines the data input capabilities of existing processes and provides extended flexibility in allowing the output to multiple files and also the re-formatting of data. The following processes allow the user to input fixed or free format data interactively or from an ASCII system file:

* [INDATA](<indata.md>)

Free format data input (maximum of 80 character line width).

* [INPUTW](<inputw.md>)

Fixed format data input (maximum of 240 character line width).

The format supplied by the user in these processes are fixed for every record read from the system file, and records not matching the prescribed format are rejected.

The reading of the input system file in **SURLOG** is controlled by user-defined input format files.

### Input Files:

### Output Files:

### Fields:

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('surlog')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute surlog
print("Running surlog...")
dm_fil.surlog(
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("surlog execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_surlog_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")